# Temperature data workflow

Frankfurt–München · Input and review snapshot: 16 October 2026

One workflow: **read observations → match primary/backup stations → screen → apply recorded review decisions → validate and export → report**.

The final output is `observations/railway_temperature_final.parquet`, with one row per railway station and UTC hour. Source observations, the initial selection and decision logs are retained for traceability. The single consolidated report is presented at the end and saved as `quality/temperature/temperature_report.md`.

Requirements: Python 3, pandas, numpy, matplotlib and pyarrow. Extract the complete execution package, preserve its folder structure, and run all six code cells in order. If needed, install dependencies in a separate cell with `%pip install pandas numpy matplotlib pyarrow`.

## 1. Setup and rules

Use the matching station mapping, source manifest, 40 original ZIPs and saved manual-review registry. Default paths work from the extracted package folder; `WEATHER_DIR` and `ARCHIVE_DIR` can be configured below.

- Historical rows take priority over recent rows within each DWD station. `-999`, blanks and non-numeric/non-finite values are unavailable; raw tokens are preserved. Conflicting values at the same archive priority are held for review.
- At each railway station-hour, use the primary observation; if unavailable, use the mapped backup at the same UTC hour. Missing values remain null if neither supplies an observation.
- Screen outside −35 to 45 °C, hourly changes greater than 10 °C, identical runs of at least 12 hours, and primary/backup differences greater than 10 °C. These are project review rules, not DWD deletion thresholds. Numeric review candidates remain unchanged until a recorded decision is applied.
- Preserve quality codes and provenance. G10, G15, G17, G20 and G21 use the backup at their flagged hours; the other 16 groups retain the primary. No interpolation or zero filling is used.

Coverage statistics use the study interval only. Exports also include a preceding 48-hour buffer for local departure-time alignment and lookbacks up to 24 hours.

In [1]:
from pathlib import Path
from urllib.parse import urlparse
import csv
import hashlib
import io
import json
import os
import zipfile
import pandas as pd
from IPython.display import display
WEATHER_DIR = None
search_roots = [Path.cwd(), *Path.cwd().parents]
if WEATHER_DIR is not None:
    mapping_candidates = [Path(WEATHER_DIR).expanduser() / 'station_mapping.json']
else:
    mapping_candidates = [candidate for base in search_roots for candidate in (base / 'station_mapping.json', base / 'frankfurt_munich/weather/station_mapping.json')]
MAPPING_PATH = next((p for p in mapping_candidates if p.is_file()), None)
if MAPPING_PATH is None:
    raise FileNotFoundError(f'station_mapping.json was not found. Kernel working directory: {Path.cwd()}. Open the notebook inside the extracted run-package folder, or set WEATHER_DIR to the folder containing the matching station_mapping.json and source_manifest.json.')
MANIFEST_PATH = MAPPING_PATH.with_name('source_manifest.json')
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'Missing companion source_manifest.json: {MANIFEST_PATH}')
mapping = json.loads(MAPPING_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
if mapping.get('metadata', {}).get('schema_version', 0) < 2:
    raise ValueError('The mapping file is an older version. Use the matching input files from the latest run package.')
expected_mapping_hash = manifest.get('supporting_file_sha256', {}).get('station_mapping.json')
if hashlib.sha256(MAPPING_PATH.read_bytes()).hexdigest() != expected_mapping_hash:
    raise ValueError('Mapping and source manifest versions do not match. Use the paired files from the run package.')
ARCHIVE_DIR = Path(os.environ.get('DWD_TEMPERATURE_ARCHIVE_DIR', str(MAPPING_PATH.parent / 'raw/air_temperature'))).expanduser()
if not ARCHIVE_DIR.is_dir():
    raise FileNotFoundError(f'Temperature archive folder not found: {ARCHIVE_DIR}. Extract the complete run package, or set ARCHIVE_DIR to the folder containing the required ZIPs.')
decision_file = MAPPING_PATH.parent / 'quality/temperature/railway_initial/manual_review_decisions.json'
if not decision_file.is_file():
    raise FileNotFoundError('Missing manual_review_decisions.json for the recorded-review step. Keep the complete run-package folder structure.')
temperature_mappings = [r for r in mapping['mappings'] if r['variable'] == 'TT_TU']
assert temperature_mappings, 'No TT_TU mappings were found.'
STUDY_START = pd.Timestamp(mapping['metadata']['period_start'], tz='UTC')
STUDY_END = pd.Timestamp(mapping['metadata']['period_end_exclusive'], tz='UTC')
BUFFER_HOURS = 48
READ_START = STUDY_START - pd.Timedelta(hours=BUFFER_HOURS)

station_links = []
required_archives = {}
for record in temperature_mappings:
    for role in ('primary', 'backup'):
        station_id = str(record[f'{role}_station_id']).zfill(5)
        station_links.append({'railway_station': record['railway_station'], 'role': role, 'dwd_station_id': station_id, 'dwd_station_name': record[f'{role}_station_name']})
        for filename in record[f'{role}_source_archives']:
            if filename in required_archives:
                assert required_archives[filename] == station_id
            required_archives[filename] = station_id
station_links = pd.DataFrame(station_links)
station_list = station_links.groupby(['dwd_station_id', 'dwd_station_name'], as_index=False).agg(roles=('role', lambda s: ', '.join(sorted(set(s)))), railway_station_count=('railway_station', 'nunique'))
source_lookup = {}
for entry in manifest['dwd_archives']:
    filename = Path(urlparse(entry['url']).path).name
    if filename in required_archives:
        if filename in source_lookup:
            assert source_lookup[filename]['sha256'] == entry['sha256'], filename
        source_lookup[filename] = entry
inventory_rows = []
for filename, station_id in sorted(required_archives.items()):
    path = ARCHIVE_DIR / filename
    source = source_lookup.get(filename)
    exists = path.is_file()
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest() if exists else None
    inventory_rows.append({'dwd_station_id': station_id, 'archive_kind': 'historical' if filename.endswith('_hist.zip') else 'recent', 'archive': filename, 'file_found': exists, 'manifest_found': source is not None, 'hash_matches': bool(source and exists and (actual_hash == source['sha256'])), 'sha256': actual_hash, 'source_url': source['url'] if source else None, 'downloaded_utc': source.get('downloaded_utc') if source else None})
archive_inventory = pd.DataFrame(inventory_rows)
bad_files = archive_inventory.loc[~archive_inventory[['file_found', 'manifest_found', 'hash_matches']].all(axis=1)]
if not bad_files.empty:
    raise ValueError('Missing or changed input files. Resolve the inventory issues before continuing.')

print(f"Inputs verified: {len(temperature_mappings)} railway stations, "
      f"{len(station_list)} DWD stations, {len(archive_inventory)} hash-matched ZIPs.")

Inputs verified: 19 railway stations, 20 DWD stations, 40 hash-matched ZIPs.


## 2. Read and standardize DWD observations

Read the product TXT members directly from the ZIPs. Check station IDs, timestamps, temperature units and UTC metadata; retain the study interval and buffer. Resolve archive overlaps, construct the DWD hourly grid and compute source-level monthly coverage, gaps, conflicts, screening flags and quality-code distributions.

These source-level checks support the railway dataset. Their detailed results are saved as audit tables, without a separate notebook report.

In [2]:
REQUIRED_COLUMNS = ['STATIONS_ID', 'MESS_DATUM', 'QN_9', 'TT_TU']

def check_parameter_metadata(archive):
    members = [n for n in archive.namelist() if Path(n).name.startswith('Metadaten_Parameter_tu_stunde_') and n.lower().endswith('.txt')]
    if len(members) != 1:
        raise ValueError('Expected one temperature parameter metadata TXT file.')
    text = archive.read(members[0]).decode('latin1')
    lines = text.splitlines()
    header = [x.strip() for x in next(csv.reader([lines[0]], delimiter=';'))]
    rows = []
    for values in csv.reader(lines[1:], delimiter=';'):
        row = {key: value.strip() for key, value in zip(header, values)}
        if row.get('Parameter') != 'TT_TU':
            continue
        start = pd.to_datetime(row.get('Von_Datum'), format='%Y%m%d', errors='raise', utc=True)
        end = pd.to_datetime(row.get('Bis_Datum'), format='%Y%m%d', errors='raise', utc=True)
        if start < STUDY_END and end + pd.Timedelta(days=1) > READ_START:
            rows.append(row)
    units_ok = bool(rows) and all((row.get('Einheit') == '°C' for row in rows))
    utc_ok = bool(rows) and all(('Stundenwerte in UTC' in row.get('Zusatz-Info', '') for row in rows))
    if not (units_ok and utc_ok):
        raise ValueError(f'Review time/units metadata before reading: {members[0]}')
    return (members[0], units_ok, utc_ok)
raw_parts = []
read_log = []
for item in archive_inventory.to_dict('records'):
    path = ARCHIVE_DIR / item['archive']
    with zipfile.ZipFile(path) as archive:
        metadata_member, units_ok, utc_ok = check_parameter_metadata(archive)
        members = [n for n in archive.namelist() if Path(n).name.startswith('produkt_tu_stunde_') and n.endswith('.txt')]
        if len(members) != 1:
            raise ValueError(f'Expected one observation TXT file: {path.name}')
        total_rows = kept_rows = 0
        first_time = last_time = None
        with archive.open(members[0]) as handle:
            for chunk in pd.read_csv(handle, sep=';', encoding='latin1', dtype='string', keep_default_na=False, chunksize=100000):
                chunk.columns = chunk.columns.str.strip()
                missing_columns = set(REQUIRED_COLUMNS) - set(chunk.columns)
                if missing_columns:
                    raise ValueError(f'Missing columns {missing_columns}: {path.name}')
                chunk = chunk[REQUIRED_COLUMNS].copy()
                for column in REQUIRED_COLUMNS:
                    chunk[column] = chunk[column].str.strip()
                station_ids = chunk['STATIONS_ID'].str.zfill(5)
                if not station_ids.eq(item['dwd_station_id']).all():
                    raise ValueError(f'Unexpected station ID: {path.name}')
                labels = chunk['MESS_DATUM']
                if not labels.str.fullmatch('\\d{10}').all():
                    raise ValueError(f'Malformed timestamp label: {path.name}')
                times = pd.to_datetime(labels, format='%Y%m%d%H', errors='raise', utc=True)
                total_rows += len(chunk)
                if len(chunk):
                    first_time = times.min() if first_time is None else min(first_time, times.min())
                    last_time = times.max() if last_time is None else max(last_time, times.max())
                keep = (times >= READ_START) & (times < STUDY_END)
                part = chunk.loc[keep].rename(columns={'STATIONS_ID': 'station_id_raw', 'MESS_DATUM': 'timestamp_label_raw', 'TT_TU': 'temperature_raw_token', 'QN_9': 'quality_code_raw'}).copy()
                if len(part):
                    part['dwd_station_id'] = item['dwd_station_id']
                    part['timestamp_utc'] = times.loc[keep]
                    part['source_kind'] = item['archive_kind']
                    part['source_archive'] = path.name
                    part['source_member'] = members[0]
                    raw_parts.append(part)
                kept_rows += len(part)
        read_log.append({'dwd_station_id': item['dwd_station_id'], 'source_kind': item['archive_kind'], 'archive': path.name, 'archive_rows': total_rows, 'archive_first_utc': first_time, 'archive_last_utc': last_time, 'retained_rows': kept_rows, 'units_celsius_verified': units_ok, 'utc_metadata_verified': utc_ok})
if not raw_parts:
    raise ValueError('No observations fall within the configured read interval.')
raw_temperature = pd.concat(raw_parts, ignore_index=True)
raw_temperature['dwd_station_id'] = raw_temperature['dwd_station_id'].astype('string')
read_log = pd.DataFrame(read_log)

station_summary = raw_temperature.groupby('dwd_station_id', as_index=False).agg(raw_rows=('timestamp_utc', 'size'), distinct_observed_hours=('timestamp_utc', 'nunique'), first_retained_utc=('timestamp_utc', 'min'), last_retained_utc=('timestamp_utc', 'max'), raw_minus999_rows=('temperature_raw_token', lambda s: int(s.eq('-999').sum())), raw_blank_rows=('temperature_raw_token', lambda s: int(s.eq('').sum())))
station_summary['extra_rows_on_repeated_hour_keys'] = station_summary['raw_rows'] - station_summary['distinct_observed_hours']
station_summary = station_list.merge(station_summary, on='dwd_station_id', validate='one_to_one')
expected_ids = set(station_list.dwd_station_id)
assert set(raw_temperature.dwd_station_id) == expected_ids
assert len(read_log) == len(archive_inventory)
assert read_log[['units_celsius_verified', 'utc_metadata_verified']].all().all()
assert raw_temperature.timestamp_utc.ge(READ_START).all()
assert raw_temperature.timestamp_utc.lt(STUDY_END).all()
assert raw_temperature.dwd_station_id.str.fullmatch('\\d{5}').all()

import numpy as np
import matplotlib.pyplot as plt
import platform
import pyarrow
REVIEW_LOW_C, REVIEW_HIGH_C = (-35.0, 45.0)
REVIEW_JUMP_C = 10.0
REVIEW_CONSTANT_HOURS = 12
ABSOLUTE_ZERO_C = -273.15
OUTPUT_DIR = MAPPING_PATH.parent / 'observations'
QC_DIR = MAPPING_PATH.parent / 'quality/temperature'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
KEYS = ['dwd_station_id', 'timestamp_utc']

def prepare_values(frame):
    result = frame.copy()
    token = result['temperature_raw_token'].astype('string').str.strip()
    numeric = pd.to_numeric(token, errors='coerce').astype('Float64')
    result['temperature_raw_c'] = numeric.where(np.isfinite(numeric).fillna(False))
    result['missing_marker'] = numeric.eq(-999).fillna(False)
    result['blank_value'] = token.isna() | token.eq('').fillna(False)
    result['parse_error'] = numeric.isna() & ~result['blank_value']
    result['nonfinite'] = (~np.isfinite(numeric)).fillna(False)
    result['below_absolute_zero'] = numeric.lt(ABSOLUTE_ZERO_C).fillna(False) & ~result['missing_marker']
    result['temperature_c'] = numeric.mask(result[['missing_marker', 'blank_value', 'parse_error', 'nonfinite']].any(axis=1))
    result['value_key'] = result['temperature_c'].astype('string').fillna('<missing>')
    result['priority'] = result['source_kind'].map({'historical': 0, 'recent': 1})
    if result['priority'].isna().any():
        raise ValueError('Unknown archive priority; review before resolving duplicates.')
    return result

def resolve_observations(frame):
    ordered = frame.sort_values(KEYS + ['priority', 'source_archive', 'source_member'], kind='stable')
    top_priority = ordered.groupby(KEYS)['priority'].transform('min')
    top = ordered.loc[ordered.priority.eq(top_priority)]
    top_stats = top.groupby(KEYS).agg(winning_priority_value_count=('value_key', 'nunique'), winning_priority_quality_count=('quality_code_raw', 'nunique'))
    duplicate_stats = ordered.groupby(KEYS).agg(raw_row_count=('value_key', 'size'), distinct_value_count=('value_key', 'nunique'), distinct_quality_code_count=('quality_code_raw', 'nunique'), archive_kind_count=('source_kind', 'nunique'))
    selected = ordered.drop_duplicates(KEYS, keep='first').set_index(KEYS).join(top_stats).join(duplicate_stats)
    selected['unresolved_value_conflict'] = selected.winning_priority_value_count.gt(1)
    selected['quality_code_conflict'] = selected.winning_priority_quality_count.gt(1)
    selected['temperature_c'] = selected.temperature_c.mask(selected.unresolved_value_conflict)
    selected['row_present'] = True
    duplicate_stats = duplicate_stats.join(top_stats)
    return (selected, duplicate_stats)
prepared = prepare_values(raw_temperature)
selected, duplicate_stats = resolve_observations(prepared)
duplicate_summary = duplicate_stats.loc[duplicate_stats.raw_row_count.gt(1)].reset_index()
conflict_keys = duplicate_stats.loc[duplicate_stats.distinct_value_count.gt(1)].index
duplicate_conflict_records = prepared.loc[pd.MultiIndex.from_frame(prepared[KEYS]).isin(conflict_keys)].drop(columns=['value_key', 'priority']).sort_values(KEYS + ['source_kind'])

hours = pd.date_range(READ_START, STUDY_END, freq='h', inclusive='left')
grid = pd.MultiIndex.from_product([sorted(station_list.dwd_station_id), hours], names=KEYS)
hourly = selected.reindex(grid).reset_index()
boolean_columns = ['row_present', 'missing_marker', 'blank_value', 'parse_error', 'nonfinite', 'below_absolute_zero', 'unresolved_value_conflict', 'quality_code_conflict']
for column in boolean_columns:
    hourly[column] = hourly[column].astype('boolean').fillna(False).astype(bool)
for column in ['raw_row_count', 'distinct_value_count', 'distinct_quality_code_count', 'archive_kind_count', 'winning_priority_value_count', 'winning_priority_quality_count']:
    hourly[column] = hourly[column].fillna(0).astype('int64')
hourly['usable_temperature'] = hourly.temperature_c.notna()
hourly['is_buffer'] = hourly.timestamp_utc.lt(STUDY_START)
hourly['missing_reason'] = np.select([~hourly.row_present, hourly.unresolved_value_conflict, hourly.missing_marker, hourly.blank_value, hourly.parse_error, hourly.nonfinite], ['no_row', 'unresolved_value_conflict', 'DWD_-999', 'blank_value', 'not_numeric', 'nonfinite'], default='none')
assert hourly.loc[~hourly.usable_temperature, 'missing_reason'].ne('none').all()
assert hourly.loc[hourly.usable_temperature, 'missing_reason'].eq('none').all()

def missing_runs(frame):
    records = []
    for station_id, group in frame.groupby('dwd_station_id', sort=True):
        group = group.sort_values('timestamp_utc')
        missing = group.temperature_c.isna()
        boundaries = (missing.ne(missing.shift()) | group.timestamp_utc.diff().ne(pd.Timedelta(hours=1))).cumsum()
        for _, segment in group.loc[missing].groupby(boundaries.loc[missing]):
            start, end = segment.timestamp_utc.iloc[[0, -1]]
            records.append({'dwd_station_id': station_id, 'start_utc': start, 'end_utc': end, 'missing_hours': len(segment), 'cross_month': start.strftime('%Y-%m') != end.strftime('%Y-%m')})
    return pd.DataFrame(records, columns=['dwd_station_id', 'start_utc', 'end_utc', 'missing_hours', 'cross_month'])
study = hourly.loc[~hourly.is_buffer].copy()
missing_gaps = missing_runs(study)

def add_review_flags(frame):
    frame = frame.sort_values(KEYS).copy()
    grouped = frame.groupby('dwd_station_id', sort=False)
    consecutive = grouped.timestamp_utc.diff().eq(pd.Timedelta(hours=1))
    frame['hourly_change_c'] = grouped.temperature_c.diff().where(consecutive)
    frame['review_extreme'] = (frame.temperature_c.lt(REVIEW_LOW_C) | frame.temperature_c.gt(REVIEW_HIGH_C)).fillna(False)
    frame['review_jump'] = frame.hourly_change_c.abs().gt(REVIEW_JUMP_C).fillna(False)
    frame['review_constant'] = False
    for _, group in frame.groupby('dwd_station_id', sort=False):
        values = group.temperature_c
        same = values.notna() & values.shift().notna() & values.eq(values.shift()).fillna(False) & group.timestamp_utc.diff().eq(pd.Timedelta(hours=1))
        runs = (~same).cumsum()
        sizes = group.groupby(runs).temperature_c.transform('size')
        frame.loc[group.index, 'review_constant'] = values.notna() & sizes.ge(REVIEW_CONSTANT_HOURS)
    quality = frame.quality_code_raw.astype('string')
    frame['review_quality_token'] = frame.row_present & ~quality.str.fullmatch('\\d+').fillna(False)
    frame['review_required'] = frame[['review_extreme', 'review_jump', 'review_constant', 'review_quality_token', 'quality_code_conflict']].any(axis=1)
    frame['qc_flag'] = frame.missing_reason.replace({'none': 'no_automatic_issue'}).astype('string')
    for flag in ['review_extreme', 'review_jump', 'review_constant', 'review_quality_token', 'quality_code_conflict']:
        frame.loc[frame[flag], 'qc_flag'] = frame.loc[frame[flag], 'qc_flag'] + '|' + flag
    return frame
hourly = add_review_flags(hourly)
study = hourly.loc[~hourly.is_buffer].copy()
study['month'] = study.timestamp_utc.dt.strftime('%Y-%m')
review_columns = KEYS + ['temperature_c', 'temperature_raw_token', 'quality_code_raw', 'hourly_change_c', 'qc_flag', 'source_archive']
review_records = study.loc[study.review_required, review_columns].copy()

MONTH_KEYS = ['dwd_station_id', 'month']
monthly = study.groupby(MONTH_KEYS).agg(expected_hours=('timestamp_utc', 'size'), usable_hours=('usable_temperature', 'sum'), source_row_hours=('row_present', 'sum'), minimum_c=('temperature_c', 'min'), maximum_c=('temperature_c', 'max'))
monthly['missing_hours'] = monthly.expected_hours - monthly.usable_hours
monthly['usable_rate'] = monthly.usable_hours / monthly.expected_hours
monthly['no_row_hours'] = monthly.expected_hours - monthly.source_row_hours
for flag in ['missing_marker', 'blank_value', 'parse_error', 'nonfinite', 'below_absolute_zero', 'unresolved_value_conflict', 'quality_code_conflict', 'review_extreme', 'review_jump', 'review_constant', 'review_quality_token', 'review_required']:
    monthly[flag + '_hours'] = study.groupby(MONTH_KEYS)[flag].sum()
monthly['duplicate_key_hours'] = study.assign(duplicate=study.raw_row_count.gt(1)).groupby(MONTH_KEYS).duplicate.sum()
monthly['different_raw_value_hours'] = study.assign(conflict=study.distinct_value_count.gt(1)).groupby(MONTH_KEYS).conflict.sum()
monthly['longest_gap_hours_within_month'] = 0
for key, group in study.groupby(MONTH_KEYS):
    gaps = missing_runs(group)
    monthly.loc[key, 'longest_gap_hours_within_month'] = int(gaps.missing_hours.max()) if len(gaps) else 0
monthly['longest_full_gap_touching_month'] = 0
for gap in missing_gaps.itertuples(index=False):
    for month in pd.period_range(gap.start_utc.strftime('%Y-%m'), gap.end_utc.strftime('%Y-%m'), freq='M').astype(str):
        key = (gap.dwd_station_id, month)
        monthly.loc[key, 'longest_full_gap_touching_month'] = max(monthly.loc[key, 'longest_full_gap_touching_month'], gap.missing_hours)
monthly = monthly.reset_index()
monthly.insert(2, 'variable', 'TT_TU')
quality_distribution = study.loc[study.row_present].assign(quality_code=study.loc[study.row_present, 'quality_code_raw'].fillna('<missing>')).groupby(MONTH_KEYS + ['quality_code']).size().rename('hours').reset_index()
quality_by_key = {key: json.dumps(dict(zip(group.quality_code.astype(str), group.hours.astype(int))), sort_keys=True) for key, group in quality_distribution.groupby(MONTH_KEYS)}
monthly['quality_code_counts'] = [quality_by_key.get((s, m), '{}') for s, m in zip(monthly.dwd_station_id, monthly.month)]
station_quality = study.groupby('dwd_station_id', as_index=False).agg(expected_hours=('timestamp_utc', 'size'), usable_hours=('usable_temperature', 'sum'), minimum_c=('temperature_c', 'min'), maximum_c=('temperature_c', 'max'), review_hours=('review_required', 'sum'))
station_quality['missing_hours'] = station_quality.expected_hours - station_quality.usable_hours
station_quality['usable_rate'] = station_quality.usable_hours / station_quality.expected_hours
max_gaps = missing_gaps.groupby('dwd_station_id').missing_hours.max()
station_quality['longest_gap_hours'] = station_quality.dwd_station_id.map(max_gaps).fillna(0).astype(int)
station_quality = station_list.merge(station_quality, on='dwd_station_id', validate='one_to_one')

expected_usable = {}
for record in temperature_mappings:
    for role in ('primary', 'backup'):
        metrics = record[f'{role}_metrics']
        station_id = str(metrics['station_id']).zfill(5)
        if station_id in expected_usable:
            assert expected_usable[station_id] == int(metrics['valid'])
        expected_usable[station_id] = int(metrics['valid'])
coverage_comparison = station_quality[['dwd_station_id', 'usable_hours']].copy()
coverage_comparison['mapping_snapshot_usable_hours'] = coverage_comparison.dwd_station_id.map(expected_usable)
coverage_comparison['difference'] = coverage_comparison.usable_hours - coverage_comparison.mapping_snapshot_usable_hours
assert coverage_comparison.difference.eq(0).all(), 'New coverage differs from mapping snapshot; investigate before export.'
sha_lookup = archive_inventory.set_index('archive').sha256
hourly['source_sha256'] = hourly.source_archive.map(sha_lookup).astype('string')
export_columns = KEYS + ['temperature_c', 'temperature_raw_c', 'temperature_raw_token', 'quality_code_raw', 'row_present', 'usable_temperature', 'missing_reason', 'qc_flag', 'review_required', 'review_extreme', 'review_jump', 'review_constant', 'review_quality_token', 'quality_code_conflict', 'unresolved_value_conflict', 'below_absolute_zero', 'hourly_change_c', 'raw_row_count', 'is_buffer', 'source_kind', 'source_archive', 'source_member', 'source_sha256']
temperature = hourly[export_columns].copy()
for column in ['dwd_station_id', 'temperature_raw_token', 'quality_code_raw', 'missing_reason', 'qc_flag', 'source_kind', 'source_archive', 'source_member', 'source_sha256']:
    temperature[column] = temperature[column].astype('string')
temperature = temperature.sort_values(KEYS).reset_index(drop=True)
expected_hours = len(pd.date_range(STUDY_START, STUDY_END, freq='h', inclusive='left'))
assert len(temperature) == len(station_list) * len(hours)
assert not temperature.duplicated(KEYS).any()
assert temperature.dwd_station_id.str.fullmatch('\\d{5}').all()
assert str(temperature.timestamp_utc.dtype) == 'datetime64[ns, UTC]'
assert np.isfinite(temperature.temperature_c.dropna().astype(float)).all()
assert hourly.loc[hourly.temperature_c.lt(ABSOLUTE_ZERO_C).fillna(False), 'review_extreme'].all()
assert not temperature.temperature_c.eq(-999).fillna(False).any()
assert temperature.usable_temperature.eq(temperature.temperature_c.notna()).all()
assert len(study) == len(station_list) * expected_hours
assert monthly.expected_hours.sum() == len(study)
assert monthly.missing_hours.sum() == study.temperature_c.isna().sum()
assert (monthly.usable_hours + monthly.missing_hours).eq(monthly.expected_hours).all()
assert missing_gaps.missing_hours.sum() == study.temperature_c.isna().sum()
parquet_path = OUTPUT_DIR / 'temperature.parquet'
temperature.to_parquet(parquet_path, index=False, engine='pyarrow', compression='zstd')
restored = pd.read_parquet(parquet_path, engine='pyarrow')
pd.testing.assert_frame_equal(temperature, restored)
station_quality.to_csv(QC_DIR / 'station_quality.csv', index=False)
monthly.to_csv(QC_DIR / 'monthly_quality.csv', index=False)
quality_distribution.to_csv(QC_DIR / 'quality_codes_by_month.csv', index=False)
missing_gaps.to_csv(QC_DIR / 'missing_gaps.csv', index=False)
review_records.to_csv(QC_DIR / 'review_records.csv', index=False)
duplicate_summary.to_csv(QC_DIR / 'duplicate_summary.csv', index=False)
duplicate_conflict_records.to_csv(QC_DIR / 'duplicate_value_conflicts.csv', index=False)
coverage_comparison.to_csv(QC_DIR / 'coverage_comparison.csv', index=False)

print(f"DWD source observations prepared: {len(station_list)} stations; "
      f"{int(study.usable_temperature.sum()):,} / {len(study):,} available study station-hours.")

DWD source observations prepared: 20 stations; 391,898 / 394,560 available study station-hours.


## 3. Match primary/backup stations and screen

Construct the initial railway-station table, then inspect missingness, duplicate keys, hourly changes, temperature ranges, constant runs, quality tokens, source switches and primary/backup differences. Record the magnitude and source of each issue before applying treatment decisions.

The initial table and issue lists preserve the evidence used for review. Shared DWD observations at different railway stations are not independent measurements.

In [3]:
RAIL_QC_DIR = QC_DIR / 'railway_initial'
RAIL_QC_DIR.mkdir(parents=True, exist_ok=True)
PAIR_DIFFERENCE_REVIEW_C = 10.0

def construct_railway_initial(observations, mappings):
    parts = []
    fields = ['timestamp_utc', 'temperature_raw_c', 'temperature_raw_token', 'row_present', 'quality_code_raw', 'source_kind', 'source_archive', 'source_sha256', 'missing_reason', 'review_required', 'unresolved_value_conflict', 'quality_code_conflict']
    for record in mappings:
        primary_id = str(record['primary_station_id']).zfill(5)
        backup_id = str(record['backup_station_id']).zfill(5)
        primary = observations.loc[observations.dwd_station_id.eq(primary_id), fields].add_prefix('primary_').rename(columns={'primary_timestamp_utc': 'timestamp_utc'})
        backup = observations.loc[observations.dwd_station_id.eq(backup_id), fields].add_prefix('backup_').rename(columns={'backup_timestamp_utc': 'timestamp_utc'})
        joined = primary.merge(backup, on='timestamp_utc', how='outer', validate='one_to_one')
        joined = joined.sort_values('timestamp_utc').reset_index(drop=True)
        p = pd.to_numeric(joined.primary_temperature_raw_c, errors='coerce').astype('Float64')
        b = pd.to_numeric(joined.backup_temperature_raw_c, errors='coerce').astype('Float64')
        p_available = np.isfinite(p).fillna(False) & p.ne(-999).fillna(False)
        b_available = np.isfinite(b).fillna(False) & b.ne(-999).fillna(False)
        p_conflict = joined.primary_unresolved_value_conflict.astype('boolean').fillna(False)
        b_conflict = joined.backup_unresolved_value_conflict.astype('boolean').fillna(False)
        use_primary = p_available & ~p_conflict
        use_backup = ~p_available & ~p_conflict & b_available & ~b_conflict
        out = pd.DataFrame({'timestamp_utc': joined.timestamp_utc})
        out['railway_station'] = record['railway_station']
        out['railway_eva_id'] = str(record['railway_eva_id'])
        out['primary_dwd_station_id'] = primary_id
        out['backup_dwd_station_id'] = backup_id
        out['primary_temperature_c'] = p.where(p_available)
        out['backup_temperature_c'] = b.where(b_available)
        out['primary_has_observation'] = p_available.astype(bool)
        out['backup_has_observation'] = b_available.astype(bool)
        out['primary_missing_reason'] = joined.primary_missing_reason.astype('string')
        out['backup_missing_reason'] = joined.backup_missing_reason.astype('string')
        out['primary_unresolved_conflict'] = p_conflict.astype(bool)
        out['backup_unresolved_conflict'] = b_conflict.astype(bool)
        out['source_role'] = pd.Series('missing', index=joined.index, dtype='string')
        out.loc[use_primary, 'source_role'] = 'primary'
        out.loc[use_backup, 'source_role'] = 'backup'
        out['temperature_c'] = pd.Series(pd.NA, index=joined.index, dtype='Float64')
        out.loc[use_primary, 'temperature_c'] = p.loc[use_primary]
        out.loc[use_backup, 'temperature_c'] = b.loc[use_backup]
        out['dwd_station_id'] = pd.Series(pd.NA, index=joined.index, dtype='string')
        out.loc[use_primary, 'dwd_station_id'] = primary_id
        out.loc[use_backup, 'dwd_station_id'] = backup_id
        out['source_distance_km'] = pd.Series(pd.NA, index=joined.index, dtype='Float64')
        out.loc[use_primary, 'source_distance_km'] = float(record['primary_distance_km'])
        out.loc[use_backup, 'source_distance_km'] = float(record['backup_distance_km'])
        for field in ['temperature_raw_token', 'quality_code_raw', 'source_kind', 'source_archive', 'source_sha256']:
            out[field] = pd.Series(pd.NA, index=joined.index, dtype='string')
            out.loc[use_primary, field] = joined.loc[use_primary, 'primary_' + field].astype('string')
            out.loc[use_backup, field] = joined.loc[use_backup, 'backup_' + field].astype('string')
        for source_field, output_field in [('review_required', 'source_station_review_flag'), ('quality_code_conflict', 'source_quality_code_conflict')]:
            out[output_field] = False
            out.loc[use_primary, output_field] = joined.loc[use_primary, 'primary_' + source_field].astype(bool)
            out.loc[use_backup, output_field] = joined.loc[use_backup, 'backup_' + source_field].astype(bool)
        out['missing_reason'] = 'none'
        out.loc[out.source_role.eq('missing'), 'missing_reason'] = 'both_stations_unavailable'
        out.loc[p_conflict, 'missing_reason'] = 'primary_archive_conflict_held_for_review'
        out.loc[~p_available & ~p_conflict & b_conflict, 'missing_reason'] = 'backup_archive_conflict_held_for_review'
        out['is_buffer'] = out.timestamp_utc.lt(STUDY_START)
        parts.append(out)
    return pd.concat(parts, ignore_index=True).sort_values(['railway_eva_id', 'timestamp_utc']).reset_index(drop=True)
railway_initial = construct_railway_initial(temperature, temperature_mappings)
railway_study = railway_initial.loc[~railway_initial.is_buffer].copy()

def screen_railway_initial(frame):
    out = frame.sort_values(['railway_eva_id', 'timestamp_utc']).copy()
    grouped = out.groupby('railway_eva_id', sort=False)
    consecutive = grouped.timestamp_utc.diff().eq(pd.Timedelta(hours=1))
    out['hourly_change_c'] = grouped.temperature_c.diff().where(consecutive)
    previous_source = grouped.dwd_station_id.shift()
    previous_value = grouped.temperature_c.shift()
    out['source_changed'] = (consecutive & out.temperature_c.notna() & previous_value.notna() & out.dwd_station_id.ne(previous_source).fillna(False)).astype(bool)
    out['primary_backup_difference_c'] = (out.primary_temperature_c - out.backup_temperature_c).abs()
    out['review_below_absolute_zero'] = out.temperature_c.lt(ABSOLUTE_ZERO_C).fillna(False)
    out['review_extreme'] = (out.temperature_c.lt(REVIEW_LOW_C) | out.temperature_c.gt(REVIEW_HIGH_C)).fillna(False)
    out['review_jump'] = out.hourly_change_c.abs().gt(REVIEW_JUMP_C).fillna(False)
    out['review_jump_at_source_change'] = out.review_jump & out.source_changed
    out['review_pair_difference'] = out.primary_backup_difference_c.gt(PAIR_DIFFERENCE_REVIEW_C).fillna(False)
    out['review_constant'] = False
    for _, group in out.groupby('railway_eva_id', sort=False):
        values = group.temperature_c
        same = values.notna() & values.shift().notna() & values.eq(values.shift()).fillna(False) & group.timestamp_utc.diff().eq(pd.Timedelta(hours=1))
        runs = (~same).cumsum()
        sizes = group.groupby(runs).temperature_c.transform('size')
        out.loc[group.index, 'review_constant'] = values.notna() & sizes.ge(REVIEW_CONSTANT_HOURS)
    out['review_quality_token'] = out.temperature_c.notna() & ~out.quality_code_raw.str.fullmatch('\\d+').fillna(False)
    out['review_archive_conflict'] = out.primary_unresolved_conflict | ~out.primary_has_observation & out.backup_unresolved_conflict
    flag_columns = ['source_station_review_flag', 'source_quality_code_conflict', 'review_below_absolute_zero', 'review_extreme', 'review_jump', 'review_pair_difference', 'review_constant', 'review_quality_token', 'review_archive_conflict']
    out['review_required'] = out[flag_columns].any(axis=1)
    out['issue_flags'] = out.missing_reason.replace({'none': ''}).astype('string')
    for flag in flag_columns:
        out.loc[out[flag], 'issue_flags'] = out.loc[out[flag], 'issue_flags'] + '|' + flag
    out['issue_flags'] = out.issue_flags.str.strip('|').replace({'': 'none'})
    return out
railway_initial = screen_railway_initial(railway_initial)
railway_study = railway_initial.loc[~railway_initial.is_buffer].copy()
railway_study['month'] = railway_study.timestamp_utc.dt.strftime('%Y-%m')
issue_flags = ['review_below_absolute_zero', 'review_extreme', 'review_jump', 'review_jump_at_source_change', 'review_pair_difference', 'review_constant', 'review_quality_token', 'review_archive_conflict', 'source_station_review_flag', 'source_quality_code_conflict']
problem_records = railway_study.loc[railway_study.temperature_c.isna() | railway_study.review_required].copy()

def interval_records(frame, mask_column, value=True):
    records = []
    for eva, group in frame.groupby('railway_eva_id', sort=True):
        group = group.sort_values('timestamp_utc')
        mask = group[mask_column].eq(value).fillna(False)
        boundaries = (mask.ne(mask.shift()) | group.timestamp_utc.diff().ne(pd.Timedelta(hours=1))).cumsum()
        for _, segment in group.loc[mask].groupby(boundaries.loc[mask]):
            records.append({'railway_eva_id': eva, 'railway_station': segment.railway_station.iloc[0], 'start_utc': segment.timestamp_utc.iloc[0], 'end_utc': segment.timestamp_utc.iloc[-1], 'hours': len(segment), 'cross_month': segment.timestamp_utc.iloc[0].strftime('%Y-%m') != segment.timestamp_utc.iloc[-1].strftime('%Y-%m')})
    return pd.DataFrame(records, columns=['railway_eva_id', 'railway_station', 'start_utc', 'end_utc', 'hours', 'cross_month'])
railway_study['remaining_missing'] = railway_study.temperature_c.isna()
remaining_gaps = interval_records(railway_study, 'remaining_missing')
backup_intervals = interval_records(railway_study, 'source_role', 'backup')

RKEYS = ['railway_eva_id', 'railway_station']
railway_summary = railway_study.groupby(RKEYS, as_index=False).agg(expected_hours=('timestamp_utc', 'size'), available_hours=('temperature_c', 'count'), primary_missing_hours=('primary_has_observation', lambda s: int((~s).sum())), primary_used_hours=('source_role', lambda s: int(s.eq('primary').sum())), backup_used_hours=('source_role', lambda s: int(s.eq('backup').sum())), min_temperature_c=('temperature_c', 'min'), max_temperature_c=('temperature_c', 'max'), review_hours=('review_required', 'sum'), max_abs_hourly_change_c=('hourly_change_c', lambda s: s.abs().max()), max_primary_backup_difference_c=('primary_backup_difference_c', 'max'), source_change_hours=('source_changed', 'sum'))
railway_summary['remaining_missing_hours'] = railway_summary.expected_hours - railway_summary.available_hours
railway_summary['available_rate'] = railway_summary.available_hours / railway_summary.expected_hours
railway_summary['backup_fraction'] = railway_summary.backup_used_hours / railway_summary.expected_hours
for name, intervals in [('longest_remaining_gap_hours', remaining_gaps), ('longest_backup_interval_hours', backup_intervals)]:
    longest = intervals.groupby('railway_eva_id').hours.max()
    railway_summary[name] = railway_summary.railway_eva_id.map(longest).fillna(0).astype(int)
railway_monthly = railway_study.groupby(RKEYS + ['month'], as_index=False).agg(expected_hours=('timestamp_utc', 'size'), available_hours=('temperature_c', 'count'), primary_missing_hours=('primary_has_observation', lambda s: int((~s).sum())), backup_used_hours=('source_role', lambda s: int(s.eq('backup').sum())), min_temperature_c=('temperature_c', 'min'), max_temperature_c=('temperature_c', 'max'), review_hours=('review_required', 'sum'))
railway_monthly['remaining_missing_hours'] = railway_monthly.expected_hours - railway_monthly.available_hours
railway_monthly['available_rate'] = railway_monthly.available_hours / railway_monthly.expected_hours
railway_monthly['backup_fraction'] = railway_monthly.backup_used_hours / railway_monthly.expected_hours
railway_quality_codes = railway_study.loc[railway_study.temperature_c.notna()].groupby(RKEYS + ['month', 'quality_code_raw'], dropna=False).size().rename('hours').reset_index()
inventory = [{'check': 'remaining_missing_after_backup', 'railway_station_hours': int(railway_study.remaining_missing.sum()), 'status': 'Missing values remain null; treatment pending if present.'}, {'check': 'primary_missing_covered_by_backup', 'railway_station_hours': int(railway_study.source_role.eq('backup').sum()), 'status': 'Recorded substitutions under the agreed selection rule.'}, {'check': 'source_changes', 'railway_station_hours': int(railway_study.source_changed.sum()), 'status': 'Diagnostic; a source change is not itself an error.'}]
for flag in issue_flags:
    inventory.append({'check': flag, 'railway_station_hours': int(railway_study[flag].sum()), 'status': 'Flagged observations retained; interpretation/treatment pending.'})
inventory.extend([{'check': 'duplicate_railway_station_hour_keys', 'railway_station_hours': int(railway_study.duplicated(['railway_eva_id', 'timestamp_utc']).sum()), 'status': 'Structural check.'}, {'check': 'selected_source_beyond_50km', 'railway_station_hours': int(railway_study.source_distance_km.gt(50).sum()), 'status': 'Structural check against the station-mapping distance limit.'}])
issue_inventory = pd.DataFrame(inventory)

railway_key = ['railway_eva_id', 'timestamp_utc']
assert not railway_initial.duplicated(railway_key).any()
assert len(railway_initial) == len(temperature_mappings) * len(hours)
assert len(railway_study) == len(temperature_mappings) * expected_hours
assert railway_initial.railway_eva_id.astype('string').str.fullmatch('\\d{7}').all()
for column in ['primary_dwd_station_id', 'backup_dwd_station_id']:
    assert railway_initial[column].str.fullmatch('\\d{5}').all()
assert railway_initial.dwd_station_id.dropna().str.fullmatch('\\d{5}').all()
assert railway_initial.source_distance_km.dropna().le(50).all()
assert railway_initial.temperature_c.notna().eq(railway_initial.source_role.isin(['primary', 'backup'])).all()
assert railway_initial.loc[railway_initial.source_role.eq('primary'), 'temperature_c'].eq(railway_initial.loc[railway_initial.source_role.eq('primary'), 'primary_temperature_c']).all()
assert railway_initial.loc[railway_initial.source_role.eq('backup'), 'temperature_c'].eq(railway_initial.loc[railway_initial.source_role.eq('backup'), 'backup_temperature_c']).all()
assert not railway_initial.loc[railway_initial.source_role.eq('backup'), 'primary_has_observation'].any()
assert not railway_initial.temperature_c.eq(-999).fillna(False).any()
assert railway_monthly.expected_hours.sum() == len(railway_study)
assert remaining_gaps.hours.sum() == railway_study.remaining_missing.sum()
assert backup_intervals.hours.sum() == railway_study.source_role.eq('backup').sum()
summary_lookup = railway_summary.set_index('railway_station')
for record in temperature_mappings:
    row = summary_lookup.loc[record['railway_station']]
    assert int(row.primary_missing_hours) == int(record['primary_unusable'])
    assert int(row.backup_used_hours) == int(record['recovered_by_backup'])
    assert int(row.remaining_missing_hours) == int(record['remaining_unusable'])
railway_initial_path = OUTPUT_DIR / 'railway_temperature_initial.parquet'
railway_initial.to_parquet(railway_initial_path, index=False, engine='pyarrow', compression='zstd')
pd.testing.assert_frame_equal(railway_initial, pd.read_parquet(railway_initial_path))
for filename, table in {'station_summary.csv': railway_summary, 'monthly_summary.csv': railway_monthly, 'quality_codes_by_month.csv': railway_quality_codes, 'issue_inventory.csv': issue_inventory, 'problem_records.csv': problem_records, 'remaining_gaps.csv': remaining_gaps, 'backup_intervals.csv': backup_intervals}.items():
    table.to_csv(RAIL_QC_DIR / filename, index=False)

print(f"Initial railway selection: {len(railway_study):,} study station-hours; "
      f"{int(railway_study.source_role.eq('backup').sum()):,} use backup; "
      f"{int(railway_study.temperature_c.isna().sum())} remain missing; "
      f"{int(railway_study.review_required.sum())} records flagged for review.")

Initial railway selection: 374,832 study station-hours; 2,704 use backup; 0 remain missing; 28 records flagged for review.


## 4. Apply recorded review decisions

Locate the original observations and their ±6-hour context, then apply the saved decisions using railway EVA IDs and exact UTC hours. The registry retains all 28 record decisions; G10 represents two railway stations, giving six backup substitutions.

Update the selected source, quality code, archive and distance. Recompute diagnostics at edited and neighbouring hours. A retained threshold flag remains visible with its review status. These are project source choices, not independent confirmation that the primary observations are erroneous.

In [4]:
import math
import textwrap
from zoneinfo import ZoneInfo
import matplotlib.dates as mdates

def build_temperature_review_locator(weather_dir, archive_dir):
    weather_dir, archive_dir = (Path(weather_dir), Path(archive_dir))
    output = weather_dir / 'quality/temperature/railway_initial'
    mapping = json.loads((weather_dir / 'station_mapping.json').read_text(encoding='utf-8'))
    mappings = [r for r in mapping['mappings'] if r['variable'] == 'TT_TU']
    rail_mapping = {r['railway_station']: r for r in mappings}
    order = {r['railway_station']: i for i, r in enumerate(mappings)}
    observations_path = weather_dir / 'observations/temperature.parquet'
    railway_path = weather_dir / 'observations/railway_temperature_initial.parquet'
    observations = pd.read_parquet(observations_path)
    railway = pd.read_parquet(railway_path)
    candidates = railway.loc[~railway.is_buffer & (railway.review_required | railway.temperature_c.isna())].copy()
    candidates['railway_order'] = candidates.railway_station.map(order)
    candidates = candidates.sort_values(['timestamp_utc', 'railway_order']).reset_index(drop=True)
    candidates['review_id'] = [f'R{i:02d}' for i in range(1, len(candidates) + 1)]
    assert not candidates.duplicated(['railway_eva_id', 'timestamp_utc']).any()
    names = {}
    for r in mappings:
        for role in ['primary', 'backup']:
            names[r[role + '_station_id']] = r[role + '_station_name']
    obs = observations.set_index(['dwd_station_id', 'timestamp_utc']).sort_index()
    rail = railway.set_index(['railway_eva_id', 'timestamp_utc']).sort_index()
    zone = ZoneInfo('Europe/Berlin')
    group_keys = []
    for r in candidates.itertuples(index=False):
        key = (r.timestamp_utc, tuple(sorted([r.primary_dwd_station_id, r.backup_dwd_station_id])))
        if key not in group_keys:
            group_keys.append(key)
    group_ids = {key: f'G{i:02d}' for i, key in enumerate(group_keys, 1)}
    source_targets = {}
    items = []

    def finite_value(value):
        return float(value) if pd.notna(value) and np.isfinite(float(value)) else None

    def number(value):
        return '—' if value is None or pd.isna(value) else f'{float(value):.1f}'
    for r in candidates.itertuples(index=False):
        t = r.timestamp_utc
        group_key = (t, tuple(sorted([r.primary_dwd_station_id, r.backup_dwd_station_id])))
        record = rail_mapping[r.railway_station]
        context = []
        for hour in range(-6, 7):
            hour_time = t + pd.Timedelta(hours=hour)
            p = obs.loc[r.primary_dwd_station_id, hour_time]
            b = obs.loc[r.backup_dwd_station_id, hour_time]
            selected = rail.loc[r.railway_eva_id, hour_time]
            context.append({'offset_hours': hour, 'timestamp_utc': hour_time.isoformat(), 'timestamp_germany': hour_time.tz_convert(zone).isoformat(), 'primary_temperature_c': finite_value(p.temperature_raw_c) if pd.notna(p.temperature_raw_c) and p.temperature_raw_c != -999 else None, 'backup_temperature_c': finite_value(b.temperature_raw_c) if pd.notna(b.temperature_raw_c) and b.temperature_raw_c != -999 else None, 'selected_temperature_c': finite_value(selected.temperature_c), 'selected_role': selected.source_role, 'selected_dwd_station_id': selected.dwd_station_id if pd.notna(selected.dwd_station_id) else None})
        reasons = []
        if r.review_jump:
            direction = 'Drop' if r.hourly_change_c < 0 else 'Rise'
            reasons.append(f'{direction} {abs(r.hourly_change_c):.1f} °C / 1 h')
        if r.review_pair_difference:
            reasons.append(f'Pair gap {r.primary_backup_difference_c:.1f} °C')
        for flag, label in [('review_extreme', 'Outside screening range'), ('review_below_absolute_zero', 'Below absolute zero'), ('review_constant', 'Long constant run'), ('review_quality_token', 'Quality token'), ('review_archive_conflict', 'Archive conflict')]:
            if getattr(r, flag):
                reasons.append(label)
        if r.source_station_review_flag and (not r.review_jump):
            reasons.append('Source-station review flag')
        if r.source_quality_code_conflict:
            reasons.append('Source quality-code conflict')
        if pd.isna(r.temperature_c):
            reasons.append('Missing temperature')
        item = {'review_id': r.review_id, 'plot_group': group_ids[group_key], 'review_key': f"{r.railway_eva_id}:{t.strftime('%Y%m%d%H')}", 'railway_station': r.railway_station, 'railway_eva_id': r.railway_eva_id, 'timestamp_utc': t.isoformat(), 'timestamp_germany': t.tz_convert(zone).isoformat(), 'german_time_label': t.tz_convert(zone).strftime('%Y-%m-%d %H:%M %Z'), 'primary_dwd_station_id': r.primary_dwd_station_id, 'primary_dwd_station_name': names[r.primary_dwd_station_id], 'primary_distance_km': record['primary_distance_km'], 'backup_dwd_station_id': r.backup_dwd_station_id, 'backup_dwd_station_name': names[r.backup_dwd_station_id], 'backup_distance_km': record['backup_distance_km'], 'selected_dwd_station_id': r.dwd_station_id, 'selected_role': r.source_role, 'quality_code': r.quality_code_raw, 'source_archive': r.source_archive, 'source_sha256': r.source_sha256, 'selected_temperature_c': finite_value(r.temperature_c), 'previous_temperature_c': context[5]['selected_temperature_c'], 'next_temperature_c': context[7]['selected_temperature_c'], 'primary_temperature_c': finite_value(r.primary_temperature_c), 'backup_temperature_c': finite_value(r.backup_temperature_c), 'hourly_change_c': finite_value(r.hourly_change_c), 'primary_backup_difference_c': finite_value(r.primary_backup_difference_c), 'source_changed': bool(r.source_changed), 'issue_flags': r.issue_flags, 'review_reason': '; '.join(reasons), 'decision': 'Pending', 'context': context}
        if pd.notna(r.dwd_station_id):
            source_row = obs.loc[r.dwd_station_id, t]
            item['source_member'] = source_row.source_member
            target = (str(r.source_archive), str(source_row.source_member))
            source_targets.setdefault(target, {})[t.strftime('%Y%m%d%H')] = str(r.dwd_station_id)
        items.append(item)
    raw_locations = {}
    for (filename, member), targets in source_targets.items():
        source_path = archive_dir / filename
        actual_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
        expected_hashes = {item['source_sha256'] for item in items if item['source_archive'] == filename}
        assert expected_hashes == {actual_hash}, f'Archive version differs: {filename}'
        with zipfile.ZipFile(source_path) as archive:
            with archive.open(member) as handle:
                reader = csv.reader(io.TextIOWrapper(handle, encoding='latin1'), delimiter=';')
                header = [v.strip() for v in next(reader)]
                index = {key: header.index(key) for key in ['STATIONS_ID', 'MESS_DATUM', 'TT_TU', 'QN_9']}
                for line_number, values in enumerate(reader, 2):
                    timestamp = values[index['MESS_DATUM']].strip()
                    if timestamp not in targets:
                        continue
                    sid = values[index['STATIONS_ID']].strip().zfill(5)
                    assert sid == targets[timestamp]
                    location_key = (filename, member, timestamp, sid)
                    raw_locations.setdefault(location_key, []).append({'line_number_including_header': line_number, 'temperature_token': values[index['TT_TU']].strip(), 'quality_code': values[index['QN_9']].strip()})
    for item in items:
        key = (item['source_archive'], item['source_member'], pd.Timestamp(item['timestamp_utc']).strftime('%Y%m%d%H'), item['selected_dwd_station_id'])
        matches = raw_locations[key]
        assert any((float(x['temperature_token']) == item['selected_temperature_c'] and x['quality_code'] == item['quality_code'] for x in matches))
        item['raw_txt_locations'] = matches
    pages = []
    for page_number, start in enumerate(range(0, len(group_keys), 7), 1):
        page_groups = group_keys[start:start + 7]
        columns = 2 if len(page_groups) > 1 else 1
        rows = math.ceil(len(page_groups) / columns)
        fig, axes = plt.subplots(rows, columns, figsize=(14, 3.8 * rows), squeeze=False, constrained_layout=True)
        for ax, key in zip(axes.ravel(), page_groups):
            timestamp, station_pair = key
            times = pd.date_range(timestamp - pd.Timedelta(hours=6), timestamp + pd.Timedelta(hours=6), freq='h')
            ids = [item['review_id'] for item in items if item['plot_group'] == group_ids[key]]
            railway_names = list(dict.fromkeys((item['railway_station'] for item in items if item['plot_group'] == group_ids[key])))
            for sid, color in zip(station_pair, ['#176b9d', '#b45a13']):
                values = obs.loc[(sid, slice(times[0], times[-1])), 'temperature_raw_c'].copy()
                values = values.mask(values.eq(-999))
                values = values.reset_index(level=0, drop=True).reindex(times)
                ax.plot(times, values.to_numpy(dtype=float, na_value=np.nan), marker='o', markersize=3, linewidth=1.5, color=color, label=f'DWD {sid}')
                at_event = values.loc[timestamp]
                if pd.notna(at_event):
                    ax.scatter([timestamp], [float(at_event)], s=65, facecolors='none', edgecolors=color, linewidths=1.8, zorder=4)
            ax.axvline(timestamp, color='#555555', linestyle='--', linewidth=1)
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M', tz=zone))
            title = f"{group_ids[key]} | {', '.join(ids)} | {timestamp.tz_convert(zone).strftime('%Y-%m-%d %H:%M %Z')}"
            title += '\n' + '\n'.join(textwrap.wrap(' / '.join(railway_names), width=58))
            ax.set_title(title, fontsize=10)
            ax.set_ylabel('Temperature (°C)')
            ax.tick_params(labelsize=8)
            ax.grid(alpha=0.2)
            ax.legend(fontsize=8, loc='best')
        for ax in axes.ravel()[len(page_groups):]:
            ax.set_visible(False)
        fig.suptitle('Flagged hours: original DWD observations, ±6 hours\nDashed line = review hour; all axis times = Europe/Berlin; no values changed', fontsize=12)
        filename = f'review_context_{page_number:02d}.png'
        fig.savefig(output / filename, dpi=130)
        plt.close(fig)
        pages.append((filename, [group_ids[key] for key in page_groups]))
    jumps = int(candidates.review_jump.sum())
    pair_gaps = int(candidates.review_pair_difference.sum())
    overlap = int((candidates.review_jump & candidates.review_pair_difference).sum())
    selected_unique = candidates[['dwd_station_id', 'timestamp_utc']].drop_duplicates()
    lines = ['# Temperature review candidates: locations and hourly context', '', f'Status: **{len(items)} railway station-hour records awaiting review; observations unchanged.**', '', f'The inventory contains {jumps} hourly-change flags and {pair_gaps} primary/backup-difference flags; {overlap} record has both flags. These are screening candidates, not confirmed measurement errors.', '', '## Locate an observation', '', '- `R` identifies a railway-station record in this fixed review snapshot; `G` identifies its paired-station chart.', '- Table times use **Europe/Berlin**, with CET/CEST shown explicitly. Exact UTC keys are retained in the source lookup below and in `review_context.json`.', '- The temperature sequence is **previous hour → selected hour → next hour** for the railway station. All temperatures and differences are in °C.', '- `Main / backup` gives the mapped DWD station IDs, not railway EVA IDs. The backup column is a comparison observation; it has not replaced a present primary value.', f'- In this snapshot, all {len(items)} selected observations come from the primary DWD station, and no flagged hourly change coincides with a source switch.', '- Decisions remain pending. A rapid cooling event or a spatial difference can reflect real weather; these checks alone do not determine treatment.', '', '## All review records', '', '| Record / chart | Railway station | German local time | Main / backup DWD | Previous → selected → next | Backup at review hour | Review trigger |', '|---|---|---|---|---:|---:|---|']
    for item in items:
        sequence = ' → '.join((number(item[key]) for key in ['previous_temperature_c', 'selected_temperature_c', 'next_temperature_c']))
        lines.append(f"| {item['review_id']} / {item['plot_group']} | {item['railway_station']} | {item['german_time_label']} | {item['primary_dwd_station_id']} / {item['backup_dwd_station_id']} | {sequence} | {number(item['backup_temperature_c'])} | {item['review_reason']} |")
    lines += ['', '## Shared observations and comparisons', '', f'The {len(items)} railway records use {len(selected_unique)} distinct selected DWD station-hour observations and {len(group_keys)} distinct station-pair/hour comparisons. Shared observations should be reviewed together; record counts are not counts of independent weather events.', '']
    for key in group_keys:
        group_items = [item for item in items if item['plot_group'] == group_ids[key]]
        if len(group_items) > 1:
            lines.append(f'- **{group_ids[key]}:** ' + '; '.join((f"{i['review_id']} — {i['railway_station']}" for i in group_items)) + '.')
    lines += ['', '## Hourly context charts', '', 'Each chart shows the two original DWD station series for six hours before and after the review hour. Legend IDs identify the stations; primary/backup roles are given in the record table. A pair can serve railway stations in opposite roles. Missing observations remain gaps.', '']
    for filename, groups in pages:
        lines += [f'### {groups[0]}–{groups[-1]}', '', f'![Hourly context: {groups[0]}–{groups[-1]}]({filename})', '']
    lines += ['## Exact source lookup', '', 'Line numbers below refer to the product TXT **inside the ZIP**, counting the header as line 1. Archive hashes, station IDs, UTC timestamps, raw temperature tokens and quality codes were checked against the exported observations. Complete ZIP/member filenames, source station names/distances and the ±6-hour context are retained in [review_context.json](review_context.json).', '', '| Record | UTC hour | DWD ID | Archive | Product TXT line |', '|---|---|---|---|---:|']
    for item in items:
        utc_label = pd.Timestamp(item['timestamp_utc']).strftime('%Y-%m-%d %H:%M')
        source_kind = 'historical' if item['source_archive'].endswith('_hist.zip') else 'recent'
        line_numbers = ', '.join((str(x['line_number_including_header']) for x in item['raw_txt_locations']))
        lines.append(f"| {item['review_id']} | {utc_label} | {item['selected_dwd_station_id']} | {source_kind} | {line_numbers} |")
    lines += ['', '## Review decisions', '', 'Inspect the context and source metadata before deciding to retain, remove or replace a record. All decisions are currently **Pending**. The raw ZIPs, source observation table and railway initial table have not been edited by this review-location step.', '', 'Inputs: [railway initial dataset](../../../observations/railway_temperature_initial.parquet), [DWD source observations](../../../observations/temperature.parquet), [issue inventory](issue_inventory.csv), and [screening report](initial_quality_report.md).', '']
    (output / 'review_candidates.md').write_text('\n'.join(lines), encoding='utf-8')
    payload = {'status': 'review_locations_prepared_no_treatment_applied', 'timezone_utc': 'UTC', 'display_timezone': 'Europe/Berlin', 'railway_input_sha256': hashlib.sha256(railway_path.read_bytes()).hexdigest(), 'dwd_input_sha256': hashlib.sha256(observations_path.read_bytes()).hexdigest(), 'counts': {'railway_records': len(items), 'selected_dwd_station_hour_observations': len(selected_unique), 'station_pair_hour_groups': len(group_keys), 'hourly_change_flags': jumps, 'pair_difference_flags': pair_gaps, 'overlapping_flags': overlap}, 'records': items}
    (output / 'review_context.json').write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    assert len(raw_locations) == len(selected_unique)
    assert all((item['decision'] == 'Pending' for item in items))
    return pd.DataFrame([{key: item[key] for key in ['review_id', 'plot_group', 'railway_station', 'german_time_label', 'selected_temperature_c', 'previous_temperature_c', 'next_temperature_c', 'backup_temperature_c', 'review_reason']} for item in items])

def apply_temperature_manual_review(weather_dir, screen_function):
    weather_dir = Path(weather_dir)
    initial_qc = weather_dir / 'quality/temperature/railway_initial'
    output_qc = weather_dir / 'quality/temperature/railway_reviewed'
    output_qc.mkdir(parents=True, exist_ok=True)
    decision_path = initial_qc / 'manual_review_decisions.json'
    decisions = json.loads(decision_path.read_text(encoding='utf-8'))
    initial_path = weather_dir / 'observations/railway_temperature_initial.parquet'
    source_path = weather_dir / 'observations/temperature.parquet'
    context_path = initial_qc / 'review_context.json'
    sha256 = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    initial_hash, source_hash = (sha256(initial_path), sha256(source_path))
    assert initial_hash == decisions['initial_table_sha256'], 'Initial table differs from the reviewed snapshot.'
    assert source_hash == decisions['dwd_source_table_sha256'], 'DWD observations differ from the reviewed snapshot.'
    assert sha256(context_path) == decisions['review_snapshot_sha256'], 'Review ID snapshot differs.'
    initial = pd.read_parquet(initial_path)
    observations = pd.read_parquet(source_path).set_index(['dwd_station_id', 'timestamp_utc'])
    mappings = {r['railway_station']: r for r in json.loads((weather_dir / 'station_mapping.json').read_text(encoding='utf-8'))['mappings'] if r['variable'] == 'TT_TU'}
    reviewed = initial.copy()
    reviewed['initial_temperature_c'] = initial.temperature_c
    reviewed['selection_reason'] = reviewed.source_role.map({'primary': 'primary_observation_available', 'backup': 'primary_unavailable_same_hour_backup', 'missing': 'both_sources_unavailable'}).astype('string')
    reviewed['manual_review_id'] = pd.Series(pd.NA, index=reviewed.index, dtype='string')
    reviewed['manual_review_group'] = pd.Series(pd.NA, index=reviewed.index, dtype='string')
    reviewed['manual_review_decision'] = pd.Series(pd.NA, index=reviewed.index, dtype='string')
    reviewed['manual_source_override'] = False
    indexed_rows = reviewed.reset_index().set_index(['railway_eva_id', 'timestamp_utc'])['index']
    assert indexed_rows.index.is_unique
    records = decisions['records']
    assert len({(r['railway_eva_id'], r['timestamp_utc']) for r in records}) == len(records)
    applied_rows = []
    override_indices = []
    for decision in records:
        time = pd.Timestamp(decision['timestamp_utc'])
        index = int(indexed_rows.loc[decision['railway_eva_id'], time])
        old = initial.loc[index]
        action = decision['decision']
        assert action in {'keep_primary', 'use_backup'}
        assert old.railway_station == decision['railway_station']
        assert old.source_role == 'primary'
        assert old.primary_dwd_station_id == decision['primary_dwd_station_id']
        assert old.backup_dwd_station_id == decision['backup_dwd_station_id']
        assert float(old.temperature_c) == decision['selected_temperature_c']
        assert float(old.primary_temperature_c) == decision['primary_temperature_c']
        assert float(old.backup_temperature_c) == decision['backup_temperature_c']
        reviewed.loc[index, 'manual_review_id'] = decision['review_id']
        reviewed.loc[index, 'manual_review_group'] = decision['plot_group']
        reviewed.loc[index, 'manual_review_decision'] = action
        if action == 'use_backup':
            sid = decision['backup_dwd_station_id']
            source = observations.loc[sid, time]
            assert pd.notna(source.temperature_raw_c) and np.isfinite(float(source.temperature_raw_c))
            assert float(source.temperature_raw_c) != -999 and (not source.unresolved_value_conflict)
            assert float(source.temperature_raw_c) == decision['backup_temperature_c']
            reviewed.loc[index, 'temperature_c'] = float(source.temperature_raw_c)
            reviewed.loc[index, 'dwd_station_id'] = sid
            reviewed.loc[index, 'source_role'] = 'backup'
            reviewed.loc[index, 'source_distance_km'] = float(mappings[old.railway_station]['backup_distance_km'])
            for field in ['temperature_raw_token', 'quality_code_raw', 'source_kind', 'source_archive', 'source_sha256']:
                reviewed.loc[index, field] = source[field]
            reviewed.loc[index, 'source_station_review_flag'] = bool(source.review_required)
            reviewed.loc[index, 'source_quality_code_conflict'] = bool(source.quality_code_conflict)
            reviewed.loc[index, 'selection_reason'] = 'reviewer_selected_same_hour_backup'
            reviewed.loc[index, 'manual_source_override'] = True
            override_indices.append(index)
        else:
            source = observations.loc[old.primary_dwd_station_id, time]
            reviewed.loc[index, 'selection_reason'] = 'reviewer_retained_primary'
        current = reviewed.loc[index]
        applied_rows.append({'review_id': decision['review_id'], 'plot_group': decision['plot_group'], 'railway_station': old.railway_station, 'railway_eva_id': old.railway_eva_id, 'timestamp_utc': time.isoformat(), 'german_local_time': time.tz_convert('Europe/Berlin').strftime('%Y-%m-%d %H:%M %Z'), 'decision': action, 'initial_dwd_station_id': old.dwd_station_id, 'reviewed_dwd_station_id': current.dwd_station_id, 'initial_temperature_c': float(old.temperature_c), 'reviewed_temperature_c': float(current.temperature_c), 'temperature_change_c': float(current.temperature_c - old.temperature_c), 'reviewed_source_distance_km': float(current.source_distance_km), 'quality_code': current.quality_code_raw, 'source_archive': current.source_archive, 'source_member': source.source_member, 'source_sha256': current.source_sha256, 'decision_basis': 'Project reviewer source selection; no independent error confirmation.'})
    reviewed = screen_function(reviewed)
    reviewed['diagnostic_flagged'] = reviewed.review_required.astype(bool)
    has_decision = reviewed.manual_review_decision.notna()
    reviewed['review_status'] = pd.Series('not_flagged', index=reviewed.index, dtype='string')
    reviewed.loc[reviewed.diagnostic_flagged & ~has_decision, 'review_status'] = 'pending_review'
    reviewed.loc[reviewed.manual_review_decision.eq('keep_primary').fillna(False), 'review_status'] = 'reviewed_keep_primary'
    reviewed.loc[reviewed.manual_review_decision.eq('use_backup').fillna(False), 'review_status'] = 'reviewed_use_backup'
    reviewed['review_required'] = reviewed.diagnostic_flagged & ~has_decision
    applied = pd.DataFrame(applied_rows)
    reviewed_study = reviewed.loc[~reviewed.is_buffer].copy()
    reviewed_study['month'] = reviewed_study.timestamp_utc.dt.strftime('%Y-%m')
    pending = reviewed_study.loc[reviewed_study.review_required].copy()
    diagnostic_records = reviewed_study.loc[reviewed_study.diagnostic_flagged].copy()
    applied['post_review_diagnostic_flagged'] = [bool(reviewed.loc[int(indexed_rows.loc[r['railway_eva_id'], pd.Timestamp(r['timestamp_utc'])])].diagnostic_flagged) for r in records]
    pd.testing.assert_index_equal(initial.index, reviewed.index)
    pd.testing.assert_series_equal(initial.timestamp_utc, reviewed.timestamp_utc)
    pd.testing.assert_series_equal(initial.railway_eva_id, reviewed.railway_eva_id)
    temperature_changed = initial.temperature_c.ne(reviewed.temperature_c).fillna(False)
    source_changed = initial.dwd_station_id.ne(reviewed.dwd_station_id).fillna(False)
    expected_changed = pd.Series(False, index=initial.index)
    expected_changed.loc[override_indices] = True
    assert temperature_changed.equals(expected_changed.astype(temperature_changed.dtype))
    assert source_changed.equals(expected_changed.astype(source_changed.dtype))
    assert int(reviewed.manual_source_override.sum()) == len(override_indices)
    assert not reviewed.duplicated(['railway_eva_id', 'timestamp_utc']).any()
    assert reviewed.temperature_c.isna().sum() == initial.temperature_c.isna().sum()
    assert reviewed.source_distance_km.dropna().le(50).all()
    assert sha256(initial_path) == initial_hash and sha256(source_path) == source_hash
    return (reviewed, applied, decisions)

review_index = build_temperature_review_locator(MAPPING_PATH.parent, ARCHIVE_DIR)
reviewed, applied_decisions, decision_registry = apply_temperature_manual_review(
    MAPPING_PATH.parent, screen_railway_initial
)
print(f"Review decisions applied: {len(applied_decisions)} records; "
      f"{int(applied_decisions.decision.eq('keep_primary').sum())} retain primary; "
      f"{int(applied_decisions.decision.eq('use_backup').sum())} use the same-hour backup.")

Review decisions applied: 28 records; 22 retain primary; 6 use the same-hour backup.


## 5. Validate and save final outputs

Check unique railway-station/hour keys, UTC timestamps, five-digit DWD IDs, selected-source distances and missingness. Verify that only the six approved temperature/source selections changed and that each value and its metadata match the selected same-hour DWD observation. Round-trip the final Parquet and save the audit tables, review log and processing manifest.

`diagnostic_flagged` retains screening evidence; `review_required` identifies outstanding decisions. Both are retrospective quality diagnostics, not prediction-time model features.

In [5]:
REVIEWED_QC_DIR = QC_DIR / "railway_reviewed"
REVIEWED_QC_DIR.mkdir(parents=True, exist_ok=True)
reviewed_study = reviewed.loc[~reviewed.is_buffer].copy()
reviewed_study["month"] = reviewed_study.timestamp_utc.dt.strftime("%Y-%m")
reviewed_path = OUTPUT_DIR / "railway_temperature_final.parquet"

assert len(reviewed) == len(railway_initial)
assert not reviewed.duplicated(["railway_eva_id", "timestamp_utc"]).any()
assert str(reviewed.timestamp_utc.dtype) == "datetime64[ns, UTC]"
assert reviewed.dwd_station_id.dropna().str.fullmatch(r"\d{5}").all()
assert reviewed.source_distance_km.dropna().le(50).all()
assert not reviewed.temperature_c.eq(-999).fillna(False).any()
changed = reviewed.manual_source_override
assert int(changed.sum()) == sum(r["decision"] == "use_backup" for r in decision_registry["records"])
pd.testing.assert_series_equal(
    railway_initial.loc[~changed, "temperature_c"], reviewed.loc[~changed, "temperature_c"])
source_index = temperature.set_index(["dwd_station_id", "timestamp_utc"])
for row in reviewed.loc[changed].itertuples(index=False):
    source = source_index.loc[(row.dwd_station_id, row.timestamp_utc)]
    assert float(row.temperature_c) == float(source.temperature_raw_c)
    assert row.quality_code_raw == source.quality_code_raw
    assert row.source_archive == source.source_archive and row.source_sha256 == source.source_sha256
assert reviewed.temperature_c.isna().sum() == railway_initial.temperature_c.isna().sum()

reviewed.to_parquet(reviewed_path, index=False, engine="pyarrow", compression="zstd")
pd.testing.assert_frame_equal(reviewed, pd.read_parquet(reviewed_path))
pending_records = reviewed_study.loc[reviewed_study.review_required].copy()
retained_diagnostics = reviewed_study.loc[reviewed_study.diagnostic_flagged].copy()
final_station_quality = reviewed_study.groupby(["railway_eva_id", "railway_station"], as_index=False).agg(
    expected_hours=("timestamp_utc", "size"), available_hours=("temperature_c", "count"),
    primary_hours=("source_role", lambda s: int(s.eq("primary").sum())),
    backup_hours=("source_role", lambda s: int(s.eq("backup").sum())),
    manual_backup_hours=("manual_source_override", "sum"),
    diagnostic_hours=("diagnostic_flagged", "sum"), pending_review_hours=("review_required", "sum"),
    minimum_c=("temperature_c", "min"), maximum_c=("temperature_c", "max"))
final_station_quality["remaining_missing_hours"] = final_station_quality.expected_hours - final_station_quality.available_hours
final_monthly_quality = reviewed_study.groupby(["railway_eva_id", "railway_station", "month"], as_index=False).agg(
    expected_hours=("timestamp_utc", "size"), available_hours=("temperature_c", "count"),
    backup_hours=("source_role", lambda s: int(s.eq("backup").sum())),
    manual_backup_hours=("manual_source_override", "sum"),
    diagnostic_hours=("diagnostic_flagged", "sum"), pending_review_hours=("review_required", "sum"),
    minimum_c=("temperature_c", "min"), maximum_c=("temperature_c", "max"))
final_monthly_quality["remaining_missing_hours"] = final_monthly_quality.expected_hours - final_monthly_quality.available_hours
for filename, table in {
    "review_decisions.csv": applied_decisions,
    "retained_diagnostics.csv": retained_diagnostics,
    "new_review_candidates.csv": pending_records,
    "station_summary.csv": final_station_quality,
    "monthly_summary.csv": final_monthly_quality,
}.items():
    table.to_csv(REVIEWED_QC_DIR / filename, index=False)

hash_file = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
workflow_manifest = {
    "status": "temperature_workflow_complete_recorded_review_decisions_applied",
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "variable": "TT_TU", "units": "degrees Celsius", "timezone": "UTC",
    "study_start_utc": STUDY_START.isoformat(), "study_end_exclusive_utc": STUDY_END.isoformat(),
    "read_start_utc": READ_START.isoformat(), "buffer_hours": BUFFER_HOURS,
    "railway_stations": len(temperature_mappings), "dwd_stations": len(station_list),
    "archive_count": len(archive_inventory), "output_rows": len(reviewed),
    "study_station_hours": len(reviewed_study),
    "study_primary_hours": int(reviewed_study.source_role.eq("primary").sum()),
    "study_backup_hours": int(reviewed_study.source_role.eq("backup").sum()),
    "manual_backup_hours": int(reviewed_study.manual_source_override.sum()),
    "remaining_missing_hours": int(reviewed_study.temperature_c.isna().sum()),
    "initial_review_records": int(railway_study.review_required.sum()),
    "retained_diagnostic_hours": len(retained_diagnostics), "pending_review_hours": len(pending_records),
    "keep_primary_groups": decision_registry["keep_primary_groups"],
    "use_backup_groups": decision_registry["use_backup_groups"],
    "screening_thresholds": {"low_c": REVIEW_LOW_C, "high_c": REVIEW_HIGH_C,
        "hourly_change_c": REVIEW_JUMP_C, "constant_hours": REVIEW_CONSTANT_HOURS,
        "primary_backup_difference_c": PAIR_DIFFERENCE_REVIEW_C},
    "rules": {"archive_priority": "historical before recent", "missing_marker": -999,
        "station_priority": "primary, then same-hour backup if unavailable",
        "numeric_review_candidates": "retained until a recorded decision is applied",
        "interpolation": False, "zero_fill": False, "review_scope": "specified railway station and UTC hour only"},
    "source_archives": archive_inventory.to_dict("records"),
    "time_and_unit_checks": read_log.to_dict("records"),
    "sha256": {"mapping": hash_file(MAPPING_PATH), "source_manifest": hash_file(MANIFEST_PATH),
        "manual_review_registry": hash_file(RAIL_QC_DIR / "manual_review_decisions.json"),
        "dwd_observations": hash_file(parquet_path), "railway_initial": hash_file(railway_initial_path),
        "railway_reviewed": hash_file(reviewed_path)},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "pyarrow": pyarrow.__version__},
    "remaining_scope": ["Comprehensive instrument/site review", "Observation publication-time availability",
                         "Prediction-time journey matching", "Other weather variables"],
}
(QC_DIR / "temperature_workflow_manifest.json").write_text(
    json.dumps(workflow_manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# The review index remains the original evidence, with a link to the current consolidated report.
index_path = RAIL_QC_DIR / "review_candidates.md"
index_text = index_path.read_text(encoding="utf-8")
index_text = index_text.replace(
    "Status: **28 railway station-hour records awaiting review; observations unchanged.**",
    "Status: **original screening snapshot, before manual review**. "
    "[Applied decisions and final report](../temperature_report.md) are documented separately.")
index_text = index_text.replace(
    "- Decisions remain pending. A rapid cooling event or a spatial difference can reflect real weather; these checks alone do not determine treatment.",
    "- Values and charts here preserve the pre-review evidence. Applied source choices are documented in the final report.")
index_text = index_text.replace(
    "All decisions are currently **Pending**. The raw ZIPs, source observation table and railway initial table have not been edited by this review-location step.",
    "The context JSON preserves its pre-review Pending labels. Applied decisions are recorded in "
    "[manual_review_decisions.json](manual_review_decisions.json) and the [final report](../temperature_report.md).")
index_text = index_text.replace("[screening report](initial_quality_report.md)", "[final report](../temperature_report.md)")
index_path.write_text(index_text, encoding="utf-8")
print("Final table, decision log and audit tables saved; key, provenance and Parquet checks passed.")

Final table, decision log and audit tables saved; key, provenance and Parquet checks passed.


## 6. Report

The consolidated report below summarizes the full workflow, including the initial screening, recorded decisions, post-review checks and remaining scope. The same report is saved beside the audit outputs.

In [6]:
from IPython.display import Markdown
import re

before = railway_study
after = reviewed_study
changes = applied_decisions.loc[applied_decisions.decision.eq("use_backup")]
lines = ["# Temperature data report", "",
    "Status: **temperature observations processed and all 28 recorded review decisions applied**.", "",
    f"Inputs: {len(archive_inventory)} hash-verified DWD temperature ZIPs from {len(station_list)} weather stations, "
    f"mapped to {len(temperature_mappings)} railway stations. Temperature units and UTC metadata were checked.", "",
    f"Before railway-level backup selection, the {len(station_list)} DWD stations provide "
    f"{int(study.usable_temperature.sum()):,} of {len(study):,} source station-hours; "
    f"{int(study.temperature_c.isna().sum()):,} are missing. These source-level counts differ from the railway grid below.", "",
    f"Study interval: **{STUDY_START.strftime('%Y-%m-%d %H:%M')} UTC to "
    f"{STUDY_END.strftime('%Y-%m-%d %H:%M')} UTC (exclusive)**. "
    f"The final table contains {len(reviewed):,} rows, including {int(reviewed.is_buffer.sum()):,} preceding buffer rows. "
    "Counts below use the study interval only.", "",
    "## Results", "",
    "| Check | Initial primary/backup selection | After recorded review |",
    "|---|---:|---:|",
    f"| Railway station-hours | {len(before):,} | {len(after):,} |",
    f"| Primary selected | {int(before.source_role.eq('primary').sum()):,} | {int(after.source_role.eq('primary').sum()):,} |",
    f"| Backup selected | {int(before.source_role.eq('backup').sum()):,} | {int(after.source_role.eq('backup').sum()):,} |",
    f"| Remaining missing | {int(before.temperature_c.isna().sum())} | {int(after.temperature_c.isna().sum())} |",
    f"| Hourly-change flags (>10 °C) | {int(before.review_jump.sum())} | {int(after.review_jump.sum())} |",
    f"| Primary/backup-gap flags (>10 °C) | {int(before.review_pair_difference.sum())} | {int(after.review_pair_difference.sum())} |",
    f"| Distinct records with diagnostic flags | {int(before.review_required.sum())} | {int(after.diagnostic_flagged.sum())} |",
    f"| Outstanding review decisions | {int(before.review_required.sum())} | {int(after.review_required.sum())} |", "",
    f"Final retained temperature range: **{float(after.temperature_c.min()):.1f} to {float(after.temperature_c.max()):.1f} °C**. "
    f"Duplicate final station-hour keys: {int(after.duplicated(['railway_eva_id', 'timestamp_utc']).sum())}. "
    f"Values outside the screening range: {int(after.review_extreme.sum())}; "
    f"long constant-run flags: {int(after.review_constant.sum())}. "
    f"The longest continuous backup interval in the initial selection was {int(backup_intervals.hours.max())} hours; "
    "this describes primary-source availability, not a remaining temperature gap.", "",
    "## Recorded review decisions", "",
    "Primary observations were retained for 16 groups (22 railway station-hours). "
    "The following five groups use the same-hour backup (six railway station-hours). "
    "Overrides apply only to the listed hour; neighbouring values are unchanged.", "",
    "| Group | Railway station | German local time | Primary → backup °C | DWD primary → backup |",
    "|---|---|---|---:|---|",
]
for row in changes.itertuples(index=False):
    lines.append(f"| {row.plot_group} | {row.railway_station} | {row.german_local_time} | "
        f"{row.initial_temperature_c:.1f} → {row.reviewed_temperature_c:.1f} | "
        f"{row.initial_dwd_station_id} → {row.reviewed_dwd_station_id} |")
lines += ["", "The retained primary observations still carry their diagnostic flags and an applied review status. "
    "Project source choices do not independently confirm that an observation is physically correct or erroneous.", "",
    "## Checks and remaining scope", "",
    "Original ZIP hashes, required columns, station IDs, UTC metadata and temperature units were checked. "
    "Source-level monthly coverage, gaps, duplicate/conflicting observations and quality-code distributions "
    "were recorded. The final table passed unique-key, selected-source provenance and Parquet round-trip checks. "
    "Diagnostics were recomputed after review, including neighbouring hours; no new or unreviewed flags remain.", "",
    "The review flags are retrospective diagnostics, not prediction-time model features. "
    "Instrument/site metadata has not been comprehensively reviewed, and observation publication latency "
    "has not been established. Journey matching must use the agreed prediction-time information cutoff. "
    "Other weather variables remain outside this temperature workflow.", "",
    "## Files", "",
    "- **Current integration input:** [railway_temperature_final.parquet](../../observations/railway_temperature_final.parquet).",
    "- Source and initial tables: [DWD observations](../../observations/temperature.parquet) and "
    "[initial railway selection](../../observations/railway_temperature_initial.parquet).",
    "- [All 28 applied decisions](railway_reviewed/review_decisions.csv), "
    "[retained diagnostics](railway_reviewed/retained_diagnostics.csv), "
    "[outstanding candidates](railway_reviewed/new_review_candidates.csv), "
    "[final station summary](railway_reviewed/station_summary.csv) and "
    "[final monthly summary](railway_reviewed/monthly_summary.csv).",
    "- Source checks: [station summary](station_quality.csv), [monthly summary](monthly_quality.csv), "
    "[gaps](missing_gaps.csv), [quality codes](quality_codes_by_month.csv), "
    "[duplicates](duplicate_summary.csv) and [value conflicts](duplicate_value_conflicts.csv).",
    "- [Original review locations and context charts](railway_initial/review_candidates.md), "
    "[decision registry](railway_initial/manual_review_decisions.json) and "
    "[processing manifest](temperature_workflow_manifest.json).", "",
    "Station identifiers are strings; preserve leading zeros when reading CSV files. "
    "Raw inputs and the initial version are retained for traceability.", ""]
final_report = "\n".join(lines)
report_path = QC_DIR / "temperature_report.md"
report_path.write_text(final_report, encoding="utf-8")
def notebook_report_link(match):
    label, target = match.groups()
    if target.startswith(("http:", "https:", "#")):
        return match.group(0)
    relative = (QC_DIR / target).resolve().relative_to(MAPPING_PATH.parent.resolve()).as_posix()
    return f"[{label}]({relative})"

notebook_report = re.sub(r"\[([^\]]+)\]\(([^)]+)\)", notebook_report_link, final_report)
display(Markdown(notebook_report))

# Temperature data report

Status: **temperature observations processed and all 28 recorded review decisions applied**.

Inputs: 40 hash-verified DWD temperature ZIPs from 20 weather stations, mapped to 19 railway stations. Temperature units and UTC metadata were checked.

Before railway-level backup selection, the 20 DWD stations provide 391,898 of 394,560 source station-hours; 2,662 are missing. These source-level counts differ from the railway grid below.

Study interval: **2024-07-01 00:00 UTC to 2026-10-01 00:00 UTC (exclusive)**. The final table contains 375,744 rows, including 912 preceding buffer rows. Counts below use the study interval only.

## Results

| Check | Initial primary/backup selection | After recorded review |
|---|---:|---:|
| Railway station-hours | 374,832 | 374,832 |
| Primary selected | 372,128 | 372,122 |
| Backup selected | 2,704 | 2,710 |
| Remaining missing | 0 | 0 |
| Hourly-change flags (>10 °C) | 17 | 11 |
| Primary/backup-gap flags (>10 °C) | 12 | 12 |
| Distinct records with diagnostic flags | 28 | 22 |
| Outstanding review decisions | 28 | 0 |

Final retained temperature range: **-15.3 to 40.3 °C**. Duplicate final station-hour keys: 0. Values outside the screening range: 0; long constant-run flags: 0. The longest continuous backup interval in the initial selection was 849 hours; this describes primary-source availability, not a remaining temperature gap.

## Recorded review decisions

Primary observations were retained for 16 groups (22 railway station-hours). The following five groups use the same-hour backup (six railway station-hours). Overrides apply only to the listed hour; neighbouring values are unchanged.

| Group | Railway station | German local time | Primary → backup °C | DWD primary → backup |
|---|---|---|---:|---|
| G10 | München Hbf | 2026-05-31 15:00 CEST | 18.8 → 26.5 | 03379 → 07431 |
| G10 | München-Pasing | 2026-05-31 15:00 CEST | 18.8 → 26.5 | 03379 → 07431 |
| G15 | Stuttgart Hbf | 2026-07-16 21:00 CEST | 17.6 → 25.9 | 04928 → 06275 |
| G17 | Augsburg Hbf | 2026-07-31 18:00 CEST | 21.6 → 29.2 | 00232 → 02905 |
| G20 | Darmstadt Hbf | 2026-08-15 19:00 CEST | 24.9 → 31.9 | 00917 → 01424 |
| G21 | Ingolstadt Hbf | 2026-08-20 17:00 CEST | 15.2 → 20.0 | 02410 → 02700 |

The retained primary observations still carry their diagnostic flags and an applied review status. Project source choices do not independently confirm that an observation is physically correct or erroneous.

## Checks and remaining scope

Original ZIP hashes, required columns, station IDs, UTC metadata and temperature units were checked. Source-level monthly coverage, gaps, duplicate/conflicting observations and quality-code distributions were recorded. The final table passed unique-key, selected-source provenance and Parquet round-trip checks. Diagnostics were recomputed after review, including neighbouring hours; no new or unreviewed flags remain.

The review flags are retrospective diagnostics, not prediction-time model features. Instrument/site metadata has not been comprehensively reviewed, and observation publication latency has not been established. Journey matching must use the agreed prediction-time information cutoff. Other weather variables remain outside this temperature workflow.

## Files

- **Current integration input:** [railway_temperature_final.parquet](observations/railway_temperature_final.parquet).
- Source and initial tables: [DWD observations](observations/temperature.parquet) and [initial railway selection](observations/railway_temperature_initial.parquet).
- [All 28 applied decisions](quality/temperature/railway_reviewed/review_decisions.csv), [retained diagnostics](quality/temperature/railway_reviewed/retained_diagnostics.csv), [outstanding candidates](quality/temperature/railway_reviewed/new_review_candidates.csv), [final station summary](quality/temperature/railway_reviewed/station_summary.csv) and [final monthly summary](quality/temperature/railway_reviewed/monthly_summary.csv).
- Source checks: [station summary](quality/temperature/station_quality.csv), [monthly summary](quality/temperature/monthly_quality.csv), [gaps](quality/temperature/missing_gaps.csv), [quality codes](quality/temperature/quality_codes_by_month.csv), [duplicates](quality/temperature/duplicate_summary.csv) and [value conflicts](quality/temperature/duplicate_value_conflicts.csv).
- [Original review locations and context charts](quality/temperature/railway_initial/review_candidates.md), [decision registry](quality/temperature/railway_initial/manual_review_decisions.json) and [processing manifest](quality/temperature/temperature_workflow_manifest.json).

Station identifiers are strings; preserve leading zeros when reading CSV files. Raw inputs and the initial version are retained for traceability.
